# NLLB translator: Spanish <-> Piapoco / Achagua

Everything lives in `src/`; this notebook only drives it. Edit `src/config.py` to change
paths, language tags or hyper-parameters, and `data/raw/language_pair.csv` to add sentences.

| Step | What it does |
|---|---|
| 1 | Check the environment (device, library versions) |
| 2 | Build one two-column CSV per pair in `data/processed/` |
| 3 | Fine-tune NLLB with LoRA (skipped when the adapter already exists) |
| 4 | Try the model from Python |
| 5 | Interactive translator (`--ui plain` here, full-screen TUI in a terminal) |
| 6 | One-shot and batch translation from the command line |

Run the TUI in a real terminal, not here:

```
python -m src.cli            # TUI on a terminal, plain prompts otherwise
python -m src.cli --ui tui   # force the full-screen interface
```


## 1. Environment

In [1]:
import sys
from pathlib import Path

sys.path.append("..")

import torch
import transformers
import peft

from src.config import ADAPTER_DIR, MODEL_NAME, EPOCHS

print("project root :", Path.cwd().parent)
print("model        :", MODEL_NAME)
print("adapter dir  :", ADAPTER_DIR)
print("torch        :", torch.__version__, "| transformers:", transformers.__version__,
      "| peft:", peft.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device       :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(CPU is slow but works)")

project root : C:\Users\pc\DataspellProjects\NLLBTest1
model        : facebook/nllb-200-distilled-600M
adapter dir  : C:\Users\pc\DataspellProjects\NLLBTest1\models\nllb-spa-piapoco-achagua
torch        : 2.14.1+cu130 | transformers: 5.18.0 | peft: 0.21.2
device       : cuda NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Build the pair CSVs

`source_language_id` 1 = Spanish, 2 = Piapoco, 3 = Achagua. Both directions are written as
`spanish,<language>` and exact duplicate pairs are dropped:

- `data/processed/spanish_piapoco.csv`
- `data/processed/spanish_achagua.csv`

In [2]:
from src.build_pairs import build_pairs

pairs = build_pairs()
for language, frame in pairs.items():
    print(language, "->", frame.shape, list(frame.columns))
    display(frame)

piapoco   12 unique pairs -> C:\Users\pc\DataspellProjects\NLLBTest1\data\processed\spanish_piapoco.csv
achagua   10 unique pairs -> C:\Users\pc\DataspellProjects\NLLBTest1\data\processed\spanish_achagua.csv
raw validation_status counts: {'validated': 24, 'valiated': 1}
piapoco -> (12, 2) ['spanish', 'piapoco']


,spanish,piapoco
0,Buenos dias Para todos.,amalaka makaita
1,que rico,kayaba y yaco
2,tengo ambee,mapinua
3,el Perro esta Feliz,"auli, kakava iwowa"
4,Pinto mi casa,Nupinta nukapewa
5,asta mañana,matatawicha
6,estas Bonita,Piaka kaybechu
7,tengo sed,makala nowowa
8,ase mucho calor,Ula maini
9,mis mascota,Nu pirranai


achagua -> (10, 2) ['spanish', 'achagua']


,spanish,achagua
0,Buenos días para todos,Chita iyajini kininama
1,Qué rico,Saite iyacana
2,Tengo hambre,Nuya samuiwowa
3,El perro está feliz,Re auri saite wowa
4,Pinto mi casa,Nusakeda nubanau
5,Hasta mañana,Masta cajoscha
6,Estás bonita,Jiya saitei
7,Tengo sed,Nuya emewowa
8,Hace mucho calor,Jokemamuacana
9,Mis mascotas,Nusrra wanarra


## 3. Fine-tune

Uses LoRA on `facebook/nllb-200-distilled-600M`, registers `piapoco_Latn` and `achagua_Latn`
as new language tags (each seeded from the average of NLLB's own tags) and trains both
directions of both pairs. Set `FORCE = True` to retrain.

Batch size, LoRA rank and precision follow the device: a 6 GB GPU runs bf16 with `r=32` and
batch 8 (~4 min), the CPU path stays on `r=16`, batch 4 and fp32 (~13 min). Pass
`holdout=0.25` to keep a test split; with 22 unique pairs a model trained on everything
only reproduces what it has seen.


In [3]:
from src.train import adapter_is_trained, train

FORCE = False
print("adapter already trained:", adapter_is_trained())
train(force=FORCE)

adapter already trained: True
Adapter already trained in C:\Users\pc\DataspellProjects\NLLBTest1\models\nllb-spa-piapoco-achagua (use force=True to retrain)


WindowsPath('C:/Users/pc/DataspellProjects/NLLBTest1/models/nllb-spa-piapoco-achagua')

## 4. Use the model from Python

One `Translator` load, then as many translations as you want.

In [4]:
from src.modeling import Translator

translator = Translator.load()
print(translator.tags.as_dict())

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

{'spanish': 'spa_Latn', 'piapoco': 'piapoco_Latn', 'achagua': 'achagua_Latn'}


In [5]:
tests = [("spanish", "piapoco", "Buenos dias Para todos."),
         ("piapoco", "spanish", "kayaba y yaco"),
         ("spanish", "achagua", "Tengo sed"),
         ("achagua", "spanish", "Nuya emewowa")]

for source, target, text in tests:
    print(f"{source} -> {target}: {text!r} -> {translator.translate_one(text, source, target)!r}")

spanish -> piapoco: 'Buenos dias Para todos.' -> 'amalaka makaita'


piapoco -> spanish: 'kayaba y yaco' -> 'que rico'


spanish -> achagua: 'Tengo sed' -> 'Nuya emewowa'


achagua -> spanish: 'Nuya emewowa' -> 'Tengo sed'


## 5. Interactive translator

In a notebook this cell runs the line-based interface: type a sentence, press Enter, get the
translation on the next line. `:1`..`:4` (or `:dirs`) pick a direction, `:swap` flips it,
`:beams N` changes beam search, `:save <file>` dumps the session and `:q` leaves.

The same commands work in the full-screen TUI (`python -m src.cli` from a terminal), which
adds Ctrl+R swap, Ctrl+L clear, Ctrl+S save, history scrolling and a live status line.
`NLLB_UI=plain python -m src.cli` forces the prompt interface even in a terminal.


In [6]:
# Interactive session. A notebook kernel has no stdin, so it prints a pointer instead.
from src.cli import main

NO_INPUT = {'StdinNotImplementedError', 'EOFError'}

try:
    main()
except Exception as exc:
    if type(exc).__name__ not in NO_INPUT:
        raise
    print('This frontend cannot accept typed input:', type(exc).__name__)
    print('Run the translator in a real terminal instead:')
    print('    python -m src.cli')


NLLB translator  Spanish <-> Piapoco / Achagua
  adapter : C:\Users\pc\DataspellProjects\NLLBTest1\models\nllb-spa-piapoco-achagua
  tags    : spanish=spa_Latn, piapoco=piapoco_Latn, achagua=achagua_Latn
  data    : piapoco: 12, achagua: 10 pairs (new tags)
  chrF++  : piapoco  spanish->piapoco 100.0  piapoco->spanish 100.0  [training rows (seen by the model)]
  chrF++  : achagua  spanish->achagua 90.6  achagua->spanish 94.4  [training rows (seen by the model)]
beams: 4
Type a sentence and press Enter. Commands:   :1..:4  pick a direction        :swap   flip the current direction    :dirs   list directions        :beams N  beam search (1, 4, 8)    :save <file>  dump the session :clear    empty the screen    :help   this list              :q       quit
This frontend cannot accept typed input: StdinNotImplementedError
Run the translator in a real terminal instead:
    python -m src.cli


## 6. One-shot and batch translation

Works from a notebook cell and from a terminal (`python -m src.cli ... --help`). With
`--csv-reference` it also reports chrF++.


In [7]:
main(["--src", "spanish", "--tgt", "achagua", "--text", "Hace mucho calor"])

NLLB translator  Spanish <-> Piapoco / Achagua
  adapter : C:\Users\pc\DataspellProjects\NLLBTest1\models\nllb-spa-piapoco-achagua
  tags    : spanish=spa_Latn, piapoco=piapoco_Latn, achagua=achagua_Latn
  data    : piapoco: 12, achagua: 10 pairs (new tags)
  chrF++  : piapoco  spanish->piapoco 100.0  piapoco->spanish 100.0  [training rows (seen by the model)]
  chrF++  : achagua  spanish->achagua 90.6  achagua->spanish 94.4  [training rows (seen by the model)]
beams: 4


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model loaded on cuda (4 beams)


Hace mucho calor
  -> Jokemamuacana


In [8]:
main(["--src", "achagua", "--tgt", "spanish",
     "--file", "../data/processed/spanish_achagua.csv",
     "--csv-column", "achagua", "--csv-reference", "spanish",
     "--output", "../data/processed/predictions_achagua.csv"])

NLLB translator  Spanish <-> Piapoco / Achagua
  adapter : C:\Users\pc\DataspellProjects\NLLBTest1\models\nllb-spa-piapoco-achagua
  tags    : spanish=spa_Latn, piapoco=piapoco_Latn, achagua=achagua_Latn
  data    : piapoco: 12, achagua: 10 pairs (new tags)
  chrF++  : piapoco  spanish->piapoco 100.0  piapoco->spanish 100.0  [training rows (seen by the model)]
  chrF++  : achagua  spanish->achagua 90.6  achagua->spanish 94.4  [training rows (seen by the model)]
beams: 4


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model loaded on cuda (4 beams)


Chita iyajini kininama
  -> Buenos días para todos
Saite iyacana
  -> Qué rico
Nuya samuiwowa
  -> Tengo sed
Re auri saite wowa
  -> El perro está feliz
Nusakeda nubanau
  -> Pinto mi casa
Masta cajoscha
  -> Hasta mañana
Jiya saitei
  -> Estás bonita
Nuya emewowa
  -> Tengo sed
Jokemamuacana
  -> Hace mucho calor
Nusrra wanarra
  -> Mis mascotas

chrF++ = 94.4
wrote ../data/processed/predictions_achagua.csv
